In [ ]:
import os
import numpy as np
import rasterio
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import sys
import json

import sqlite3

rootdir = "/home/ebr/projects/release-volume-sampler"
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_003"
os.chdir(os.path.join(rootdir, "src"))

In [ ]:
# Cluster the volumes into n_clusters, use the csv for now, but might be faster to use the already
# opened db?
volumes_csv = os.path.join(rundir, 'volumes', "volumes.csv")

tri_tif = os.path.join(rundir, 'triangulation', "triangulation.tif")
bath_tif = os.path.join(rootdir, 'input', 'bathy', 'messina_001', "bathy_truncated.tif")
upstream_dict_path = os.path.join(rundir, 'triangulation',"poly_slopes.npy")

#print('Cluster_volumes')
#cluster_volumes(volumes_csv, tri_tif, bath_tif, rundir, upstream_dict_path)

In [ ]:
# Load volumes.
# Connect to the database
conn = sqlite3.connect(os.path.join(rundir, "volumes", "volumes.db"))

# Query all data from the volumes table
df = pd.read_sql_query("SELECT * FROM volumes;", conn)

# Convert the 'released' column from string to JSON
df['released'] = df['released'].apply(json.loads)


# Display the first few rows
print(df.head())

conn.close()

In [ ]:
# Load triangle data.
from rvsampler.triangulate import Triangulation

tri = Triangulation(rundir)

In [ ]:
normals, sides, areas, slopes = tri.get_normals_sides_areas_slopes()
side_normals = tri.side_normals


In [ ]:
points.shape

In [ ]:
release = df.released[4]

In [ ]:
tri.triangles[release]

In [ ]:

points_in_release = np.unique(tri.triangles[release].flatten())

In [ ]:
points_in_release

In [ ]:
tri.easting[points_in_release]

In [ ]:
mean_easting = tri.easting[points_in_release].mean()

In [ ]:
mean_easting

In [ ]:
points[df['released'][3]]

In [ ]:
points.shape

In [ ]:
df

In [ ]:
#TODO: EBR-2023-10-30: 
# I think this should be in a separate module cluster.py importimng the VolumeDatabaseHandler
# - The clustering assigns two more collumns: a cluster label (int) and a is_representative (boolean).
def cluster_volumes(volfile, trifile, bathfile, resdir, upstream_dict_path):
    print('read volumes')
    df_vol = pd.read_csv(volfile)
    # Read triangles
    print('read triangles')
    with rasterio.open(trifile) as src:
        triangles = src.read(1)  # First band
        #triangles[triangles > 60000] = 0
        bounds = src.bounds  # (left, bottom, right, top)
        tri_profile = src.profile
        transform = src.transform 
        height, width = triangles.shape

        # Create a grid of pixel coordinates
        cols, rows = np.meshgrid(np.arange(width), np.arange(height))

        # Apply the affine transform to get x and y (lon/lat or projected coords)
        lon_tri, lat_tri = transform * (cols, rows)
    # Read bathymetri
    print('read bathymetri')
    with rasterio.open(bathfile) as src:
        bathymetri = src.read(1)  # First band
        bathymetri[bathymetri > 60000] = 0
    # Gradient
    Z = bathymetri  # shape (H, W)

    # Compute gradient along both axes
    dz_dy, dz_dx = np.gradient(Z)  # dy: rows (y-axis), dx: columns (x-axis)

    # Compute gradient magnitude (slope intensity)
    gradient_magnitude = np.sqrt(dz_dx**2 + dz_dy**2)
    
    # Build a DataFrame of triangle info
    df_tri = pd.DataFrame({
        'triangle': triangles.flatten(),
        'lon': lon_tri.flatten(),
        'lat': lat_tri.flatten(),
        'z': bathymetri.flatten(),
        'grad': gradient_magnitude.flatten()
    })

    # Step 1: Compute triangle means (same as before)
    triangle_means = df_tri.groupby('triangle')[['lon', 'lat', 'z', 'grad']].mean().reset_index()

    # Step 2: Merge means into df_vol via 'seed_triangle'
    df_vol = df_vol.merge(
        triangle_means,
        how='left',
        left_on='seed_triangle',
        right_on='triangle',
        suffixes=('', '_mean')
    )

    # Step 3: Build X matrix using column names directly
    X = df_vol[['volume', 'no2d', 'lon', 'lat', 'z']].to_numpy()

    # Step 4: Remove rows with NaNs
    X = X[~np.isnan(X).any(axis=1)]

    # Filter df_vol for rows where no2d >= 0.5
    df_filtered = df_vol[df_vol['no2d'] >= 0.5]

    # Build X_filtered matrix and remove rows with NaNs
    X_filtered = df_filtered[['volume', 'no2d', 'lon', 'lat', 'z']].to_numpy()
    X_filtered = X_filtered[~np.isnan(X_filtered).any(axis=1)]
    
    # Cluster the results
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filtered)

    print('Calcualte clusters with KMEANS')
    n_clusters = 500  # choose based on your data
    kmeans = KMeans(n_clusters=n_clusters, random_state=0)
    labels = kmeans.fit_predict(X_scaled)
    centroids = kmeans.cluster_centers_

    # Reverse the scaling of the centroids
    centroids_original = scaler.inverse_transform(centroids)

    df_vol = df_vol.iloc[:len(X_scaled)].copy()
    df_vol['cluster'] = labels
    
    # Initialize a dictionary to store closest member indices
    closest_members = {}

    # Loop over each cluster
    print('Find nearest member to centroid')
    for cluster_id in range(n_clusters):
        # Get indices of points in this cluster
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Get the points in this cluster
        cluster_points = X_scaled[cluster_indices]
        
        # Get the centroid of this cluster
        centroid = centroids[cluster_id].reshape(1, -1)
        
        # Compute distances to centroid
        distances = cdist(cluster_points, centroid).flatten()
        
        # Get the index of the closest point (relative to the cluster subset)
        min_index_in_cluster = np.argmin(distances)
        
        # Get the index in the original dataset
        original_index = cluster_indices[min_index_in_cluster]
        
        # Store the result
        closest_members[cluster_id] = original_index

    # Optionally, create a DataFrame of the results
    closest_df = df_vol.loc[list(closest_members.values())].copy()
    closest_df['cluster'] = closest_df.index.map({v: k for k, v in closest_members.items()})
    
    upstream_dict = np.load(upstream_dict_path, allow_pickle=True).item()
    # Write raster volumes
    print('Write raster volumes')
    for i in range(len(closest_df)):
        idx = closest_df.index[i]
        volume = closest_df.loc[idx]
        write_raster(dict(volume), triangles, tri_profile, os.path.join(resdir,'volumes'), raster_driver='AAIGrid', crop=True)
        write_raster(dict(volume), triangles, tri_profile, os.path.join(resdir,'volumes'), raster_driver='GTiff', crop=True)
        #write_raster(dict(volume), triangles, tri_profile, os.path.join(resdir,'volumes'), raster_driver='GTiff', crop=False)
        # Calculate the boxes
        LONLO, LONHI, LATLO, LATHI = Bingclaw_gridsize(volume, upstream_dict, lon_tri, lat_tri, triangles)
        closest_df.at[idx, 'LONLO'] = LONLO
        closest_df.at[idx, 'LONHI'] = LONHI
        closest_df.at[idx, 'LATLO'] = LATLO
        closest_df.at[idx, 'LATHI'] = LATHI
    
    # Save the clusters to csv
    closest_df.to_csv(os.path.join(resdir,'volumes','Clusters.csv'), index=False)
    
    # Save also update volume list with cluster labels
    # Overskride i fremtiden tenker jeg er likesågreit
    df_vol.to_csv(os.path.join(resdir,'volumes','Volumes2.csv'), index=False)